# Modeling Workflow
### Woojin Park, Korea University
### Copyright 2026, Korea University

This notebook preserves the original experimental flow while using repository-local data paths and a configurable class-wise sample size.


In [ ]:
from pathlib import Path
import os

# Change this value for larger experiments, for example 20000 or 40000.
SAMPLES_PER_CLASS = 1000
RANDOM_STATE = 42
DEBUG_USE_TINY_MODEL = False

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CACHE_DIR = PROJECT_ROOT / '.cache'
(CACHE_DIR / 'matplotlib').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'numba').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'huggingface').mkdir(parents=True, exist_ok=True)
(CACHE_DIR / 'huggingface' / 'datasets').mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_DIR / 'matplotlib'))
os.environ.setdefault('NUMBA_CACHE_DIR', str(CACHE_DIR / 'numba'))
os.environ.setdefault('HF_HOME', str(CACHE_DIR / 'huggingface'))
os.environ.setdefault('HF_DATASETS_CACHE', str(CACHE_DIR / 'huggingface' / 'datasets'))
os.environ.setdefault('TRANSFORMERS_CACHE', str(CACHE_DIR / 'huggingface' / 'transformers'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION', 'python')

INPUT_PATH = PROJECT_ROOT / 'data' / '02_preprocessing_outputs' / 'final_preprocessed_df.csv'
RUN_NAME = f'sample_{SAMPLES_PER_CLASS}_per_class'
MODELING_INPUT_DIR = PROJECT_ROOT / 'data' / '03_modeling_inputs' / RUN_NAME
MODELING_INPUT_DIR.mkdir(parents=True, exist_ok=True)

print('INPUT_PATH:', INPUT_PATH)
print('MODELING_INPUT_DIR:', MODELING_INPUT_DIR)


## Requirements

In [ ]:
# Run this if dependencies are missing or if matplotlib/numpy import errors appear.
# Especially for errors such as: _ARRAY_API not found / numpy.core.multiarray failed to import
# After running this cell, restart the kernel and run the notebook again from the top.
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--upgrade',
    '--force-reinstall',
    '-r',
    '../requirements.txt',
])
print('Dependency install finished. Restart the kernel before continuing.')

## Import Libraries

In [ ]:
import inspect
import numpy as np
import pandas as pd
from transformers import pipeline
import matplotlib.pyplot as plt


from sklearn.utils import resample
from datasets import Features, Value, ClassLabel
from datasets import load_dataset
from datasets import Dataset

from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification

import torch
from peft import get_peft_model, LoraConfig, TaskType

import evaluate
from transformers import TrainingArguments, Trainer


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
import wandb


## UDF

In [ ]:
def mistral_preprocessing_function(examples):
    return mistral_tokenizer(examples['text'], truncation=True, max_length=MAX_LEN)


def compute_metrics(eval_pred):
    # All metrics are already predefined in the HF `evaluate` package
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")
    f1_metric= evaluate.load("f1")
    accuracy_metric = evaluate.load("accuracy")

    logits, labels = eval_pred # eval_pred is the tuple of predictions and labels returned by the model
    predictions = np.argmax(logits, axis=-1)
    precision = precision_metric.compute(predictions=predictions, references=labels, average="weighted")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="weighted")["recall"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    # The trainer is expecting a dictionary where the keys are the metrics names and the values are the scores.
    return {"precision": precision, "recall": recall, "f1-score": f1, 'accuracy': accuracy}


class WeightedCELossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # Get model's predictions
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Compute custom loss
        loss_fct = torch.nn.CrossEntropyLoss() # weight=torch.tensor([neg_weights, pos_weights], device=model.device, dtype=logits.dtype)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss



def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}


def plot_confusion_matrix(y_preds, y_true, labels):
    cm = confusion_matrix(y_true, y_preds, normalize="true")
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
    plt.title("Normalized confusion matrix")
    plt.show()

def measure_model_performance(trainer, df_input, y_labels, labels_list):
    preds_output = trainer.predict(df_input)
    print(preds_output.metrics)
    y_preds = np.argmax(preds_output.predictions, axis=1)
    plot_confusion_matrix(y_preds, y_labels, labels_list)


# Variable Setup

In [ ]:
MAX_LEN = 512
roberta_checkpoint = 'roberta-large'
mistral_checkpoint = 'pblair-basis/Mistral-7B-v0.1' # original experiment checkpoint
mistral_debug_checkpoint = 'hf-internal-testing/tiny-random-MistralForSequenceClassification'
llama_checkpoint = 'NousResearch/Llama-2-7b-hf'

if DEBUG_USE_TINY_MODEL:
    mistral_checkpoint = mistral_debug_checkpoint
    RUN_NAME = f'debug_mistral_{SAMPLES_PER_CLASS}_per_class'
    MODELING_INPUT_DIR = PROJECT_ROOT / 'data' / '03_modeling_inputs' / RUN_NAME
    MODELING_INPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() and not DEBUG_USE_TINY_MODEL else 'cpu'
print('mistral_checkpoint:', mistral_checkpoint)
print('DEVICE:', DEVICE)
print('MODELING_INPUT_DIR:', MODELING_INPUT_DIR)

## Load Dataset

In [ ]:
#### Data Preparation step.
#### input the final_preprocessed_df.csv file path here
filename = INPUT_PATH
df = pd.read_csv(filename, low_memory=False)
df.head()

In [ ]:
df.info()

In [ ]:
df.head(3)

## Data Pre-processing

In [ ]:
#### Pre-processing Step 0. Build dataframe for modeling : select only necessary features
df_pre = df[['title_with_selftext_cleaned','class_group']]
df_pre.columns = ['text','label_str']

#### Missing balue treatment & label creation
df_pre = df_pre.dropna(subset='label_str')
df_pre['label'] = df_pre.label_str.map({
    'Depression_Group': 0,
    'Neutral_Group': 1,
    'Happy_Group': 2,
})
df_pre = df_pre[['text','label']]

#### Modeling Data Preparation & sampling
df_depression = df_pre[df_pre.label==0]
df_neutral = df_pre[df_pre.label==1]
df_happy = df_pre[df_pre.label==2]


df_depression_sampled = resample(df_depression, replace=True, n_samples=SAMPLES_PER_CLASS, random_state=RANDOM_STATE) # reproducible results
df_neutral_sampled = resample(df_neutral, replace=True, n_samples=SAMPLES_PER_CLASS, random_state=RANDOM_STATE) # reproducible results
df_happy_sampled = resample(df_happy, replace=True, n_samples=SAMPLES_PER_CLASS, random_state=RANDOM_STATE) # reproducible results
df_sampled = pd.concat([df_depression_sampled, df_neutral_sampled, df_happy_sampled])

In [ ]:
#### Dataset Split (train / validation / test) : 75% / 15% / 10 % respectively
df_train_dataset, df_validation_dataset, df_test_dataset =\
  np.split(df_sampled.sample(frac=1, random_state=RANDOM_STATE), [int(.75*len(df_sampled)), int(.9*len(df_sampled))])

###### Check the train/ validation / test count after the split
print(f'train data size : {len(df_train_dataset)}')
print(f'validation data size : {len(df_validation_dataset)}')
print(f'test data size : {len(df_test_dataset)}')

###### Save it as csv for load
df_train_dataset.to_csv(MODELING_INPUT_DIR / 'train_dataset.csv', index=False)
df_validation_dataset.to_csv(MODELING_INPUT_DIR / 'validation_dataset.csv', index=False)
df_test_dataset.to_csv(MODELING_INPUT_DIR / 'test_dataset.csv', index=False)



###### Formatted data as 'DatasetDict' type for the further modeling step
class_names = ["Depression_Group", "Neutral_Group", "Happy_Group"]
sentiment_features = Features({'text': Value('string'), 'label': ClassLabel(names=class_names)}) ### feature maping from integer to class
file_dict = {'train': str(MODELING_INPUT_DIR / 'train_dataset.csv'),
             'validation': str(MODELING_INPUT_DIR / 'validation_dataset.csv'),
             'test': str(MODELING_INPUT_DIR / 'test_dataset.csv')}
dataset = load_dataset('csv', data_files=file_dict, delimiter=",", column_names=['text', 'label'], features=sentiment_features, skiprows=1)

In [ ]:
# Split the dataset into training and validation datasets
data = dataset['train'].train_test_split(train_size=0.8, seed=42)
# Rename the default "test" split to "validation"
data['val'] = data.pop("test")
# Convert the test dataframe to HuggingFace dataset and add it into the first dataset
data['test'] = dataset['test']

In [ ]:
data['train'].to_pandas().info()
data['test'].to_pandas().info()

In [ ]:
# Number of Characters
max_char = data['train'].to_pandas()['text'].str.len().max()
# Number of Words
max_words = data['train'].to_pandas()['text'].str.split().str.len().max()

In [ ]:
###### example case check
data['train'][0]

## Modeling

In [ ]:
#### Load Mistral 7B Tokenizer
###### Add prefix space to tokenize words into subwords
mistral_tokenizer = AutoTokenizer.from_pretrained(mistral_checkpoint, add_prefix_space=True)
if mistral_tokenizer.pad_token is None:
    mistral_tokenizer.pad_token = mistral_tokenizer.eos_token or mistral_tokenizer.unk_token
mistral_tokenizer.pad_token_id = mistral_tokenizer.convert_tokens_to_ids(mistral_tokenizer.pad_token)

mistral_tokenized_datasets = data.map(mistral_preprocessing_function, batched=False)
mistral_tokenized_datasets.set_format('torch')

###### Data collator for padding a batch of examples to the maximum length seen in the batch
mistral_data_collator = DataCollatorWithPadding(tokenizer=mistral_tokenizer)

In [ ]:
# Load a pre-trained model with a sequence classification header
model_load_kwargs = dict(
    pretrained_model_name_or_path=mistral_checkpoint,
    num_labels=3,
    ignore_mismatched_sizes=True,
)
if not DEBUG_USE_TINY_MODEL:
    model_load_kwargs.update(dict(device_map='auto'))

mistral_model = AutoModelForSequenceClassification.from_pretrained(**model_load_kwargs)
mistral_model.config.pad_token_id = mistral_tokenizer.pad_token_id
if DEBUG_USE_TINY_MODEL:
    mistral_model = mistral_model.to(DEVICE)

In [ ]:
######  Lora configuration step
best_parameters = globals().get('best_parameters', {
    'batch_size': 2 if DEBUG_USE_TINY_MODEL else 3,
    'epochs': 1 if DEBUG_USE_TINY_MODEL else 3,
    'lr': 2e-5,
    'lora_rank': 4,
    'lora_alpha': 16,
    'lora_dropout': 0.1,
})

mistral_model.config.pad_token_id = mistral_tokenizer.pad_token_id
mistral_peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=int(best_parameters.get('lora_rank', 4)),
    lora_alpha=int(best_parameters.get('lora_alpha', 16)),
    lora_dropout=float(best_parameters.get('lora_dropout', 0.1)),
    bias='none',
    target_modules=['q_proj', 'v_proj'],
)

mistral_model = get_peft_model(mistral_model, mistral_peft_config)
mistral_model.print_trainable_parameters()

trainable params: 864,256 || all params: 7,111,536,640 || trainable%: 0.012152872772093318

## Hyperparameter tuning (wandb)

In [ ]:
# Optional W&B sweep cell from the original experiment.
# Keep this disabled for normal/debug runs.
# Fill in your own entity/project/sweep_id before using W&B.

In [ ]:
# Optional W&B sweep cell from the original experiment.
# Keep this disabled for normal/debug runs.
# Fill in your own entity/project/sweep_id before using W&B.

In [ ]:
# Optional W&B sweep cell from the original experiment.
# Keep this disabled for normal/debug runs.
# Fill in your own entity/project/sweep_id before using W&B.

In [ ]:
######  Lora configuration step
mistral_model.config.pad_token_id = mistral_model.config.eos_token_id

mistral_peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=best_parameters['lr'], lora_alpha=best_parameters['lr'], lora_dropout=best_parameters['lr'], bias="none",
    target_modules=[
        "q_proj",
        "v_proj",
    ],
)

mistral_model = get_peft_model(mistral_model, mistral_peft_config)
mistral_model.print_trainable_parameters()

In [ ]:
###### model parameter setting
lr = best_parameters['lr']
batch_size = best_parameters['batch_size']
num_epochs = best_parameters['epochs']
output_dir = PROJECT_ROOT / 'models' / f'mistral_lora_{RUN_NAME}'

training_arg_kwargs = dict(
    output_dir=str(output_dir),
    learning_rate=lr,
    lr_scheduler_type='constant',
    warmup_ratio=0.1,
    max_grad_norm=0.3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.001,
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    fp16=torch.cuda.is_available() and not DEBUG_USE_TINY_MODEL,
    gradient_checkpointing=not DEBUG_USE_TINY_MODEL,
)
if 'eval_strategy' in inspect.signature(TrainingArguments).parameters:
    training_arg_kwargs['eval_strategy'] = 'epoch'
else:
    training_arg_kwargs['evaluation_strategy'] = 'epoch'
training_args = TrainingArguments(**training_arg_kwargs)

trainer_kwargs = dict(
    model=mistral_model,
    args=training_args,
    train_dataset=mistral_tokenized_datasets['train'],
    eval_dataset=mistral_tokenized_datasets['val'],
    data_collator=mistral_data_collator,
    compute_metrics=compute_metrics,
)
if 'processing_class' in inspect.signature(Trainer).parameters:
    trainer_kwargs['processing_class'] = mistral_tokenizer
elif 'tokenizer' in inspect.signature(Trainer).parameters:
    trainer_kwargs['tokenizer'] = mistral_tokenizer
mistral_trainer = WeightedCELossTrainer(**trainer_kwargs)

In [ ]:
mistral_trainer.train()

In [ ]:
####### evaluation metrics
metrics = mistral_trainer.evaluate(eval_dataset=mistral_tokenized_datasets["val"])
print(metrics)

In [ ]:
# Save trained model and tokenizer
mistral_trainer.save_model()
mistral_tokenizer.save_pretrained("mistral-lora-token-classification")

In [ ]:
#### Model evaluation Confuxion matrix
labels_names_list = dataset['train'].features['label'].names

###### validation set
measure_model_performance(mistral_trainer, mistral_tokenized_datasets["val"], mistral_tokenized_datasets["val"]['label'], labels_names_list)

In [ ]:
###### test set
measure_model_performance(mistral_trainer, mistral_tokenized_datasets["test"], mistral_tokenized_datasets["test"]['label'], labels_names_list)

In [ ]:
# # # Save the model and tokenizer
# # trainer.model.save_pretrained(new_model)
# # trainer.tokenizer.save_pretrained(new_model)

# from peft import LoraConfig, PeftModel

# print("Loading PEFT model")
# model = PeftModel.from_pretrained(mistral_model, "mistral-lora-token-classification")
# print(f"Running merge_and_unload")
# model = model.merge_and_unload()

# tokenizer = AutoTokenizer.from_pretrained(mistral_checkpoint)

# merged_model = "mistral-lora-token-classification/merged"
# model.save_pretrained(merged_model)
# tokenizer.save_pretrained(merged_model)
# print(f"Model saved to {merged_model}")

In [ ]:
# from peft import AutoPeftModelForSequenceClassification
# from transformers import AutoTokenizer

## Model Evaluation Demo

In [ ]:
###### Define classifier
labels_names_list = dataset['train'].features['label'].names
classifier_model = mistral_model.merge_and_unload() if hasattr(mistral_model, 'merge_and_unload') else mistral_model
classifier = pipeline('text-classification', model=classifier_model, tokenizer=mistral_tokenizer, device=0 if torch.cuda.is_available() and not DEBUG_USE_TINY_MODEL else -1)

In [ ]:
custom_tweet = "Visually stunning with some amazing battle scenes but it did have major pacing issues, especially in the first half. Also there was practically zero context given for like half the stuff that happened."
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
print(custom_tweet)

In [ ]:
custom_tweet = custom_tweet = "I failed my NLP exam yesterday and was really sad and didn't know what to do.\
I thought like life really sucks but today I met my best friend and he encouraged me and cheered me up to overcome this hardship. \
Because of his support I am not sad anymore and exam is just exam and it will be fine."
print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet = "Although there are times when I feel down at school, there are also moments of pure happiness that brighten my day.\
    It's like discovering unexpected joy in the midst of challenges, reminding me that happiness can be found even in the smallest of moments."
print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet = "Even when I'm in school, with friends and activities around, the feeling of happiness is weak compared to the sadness I often feel. \
   It's like trying to find a small spark of joy in a big storm of gloominess. Despite the efforts to cheer up, the weight of sadness seems to always linger.\
    It's like feeling a little bit of sunshine on a cloudy day, but the clouds never really go away. \
    Even with school life bustling around, the feeling of sadness is still there, making it hard to fully enjoy the moment."
print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

## Custom dataset test

## Depression

In [ ]:
custom_tweet = \
"Even though the halls are filled with laughter and chatter, there's this feeling weighing on me.\
It's like riding the wave of excitement while also carrying a bit of sadness, a reminder that life isn't always sunshine and rainbows.\
Despite the energy around me, there's a cloud in my mind, making everything seem a bit less bright.\
It's a mix of happy and sad, like a seesaw where neither side is quite up or down. \
But I still hold onto hope, believing things will get better, even if they're a bit uncertain right now."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
preds_df.transpose()

In [ ]:
custom_tweet = "During school days, there are moments when I feel a sense of happiness, perhaps when I receive praise from a teacher or when I achieve a personal goal. \
However, underlying that happiness, there's often a deeper feeling of despair, like when I struggle to keep up with my classmates or when I feel isolated in a crowd. \
Even though there are moments of joy, the weight of hopelessness seems to overshadow them, leaving me feeling more alone than ever."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet = "When I experience moments of contentment, such as when I finish a project ahead of schedule or receive positive feedback from a colleague.\
Yet, despite these brief moments of happiness, there's a persistent sense of emptiness that lingers, stemming from feelings of insecurity or dissatisfaction with my career. \
Despite my efforts to focus on the positives, the feeling of despair remains, making it difficult to fully enjoy the successes I achieve."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet = "Within my family, there are times when I share moments of laughter and love with my loved ones, cherishing the bonds we share.\
However, amidst these moments of happiness, there's an underlying feeling of sorrow that I can't seem to shake off, stemming from unresolved conflicts or distant relationships. \
Despite the warmth of family connections, the weight of melancholy prevails, casting a shadow over our interactions and leaving me yearning for deeper connection and understanding."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet = custom_tweet = \
"Even on a lively school day, amidst the laughter and chatter of friends, there's a quiet ache in my heart. It's like being part of the joy while carrying a hidden weight, \
a faint feeling of emptiness that mutes the excitement, reminding me that even in the midst of happiness, there's a subtle shadow."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet = "Even though the halls are filled with laughter and chatter, there's this feeling weighing on me.\
It's like riding the wave of excitement while also carrying a bit of sadness, a reminder that life isn't always sunshine and rainbows.\
Despite the energy around me, there's a cloud in my mind, making everything seem a bit less bright.\
It's a mix of happy and sad, like a seesaw where neither side is quite up or down. \
But I still hold onto hope, believing things will get better, even if they're a bit uncertain right now."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

## Happy

In [ ]:
custom_tweet = \
"During school days, there are moments when I feel down, maybe because of a bad grade or a disagreement with a friend. \
But amidst those gloomy times, there are also moments of happiness, like when I solve a problem I've been struggling with\
or when I share a laugh with my classmates. Even though the sad moments weigh on me, the happiness I feel is a bit stronger.\
It's like finding a small flower blooming in the midst of a barren field, a tiny glimmer of light in the darkness. \
Despite the challenges, those moments of joy keep me going, reminding me that there's always something to smile about,\
even in the toughest of times."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
preds_df.transpose()

In [ ]:
custom_tweet =  "During school days, there are moments when I feel down, maybe because of a bad grade or a disagreement with a friend. \
But amidst those gloomy times, there are also moments of happiness, like when I solve a problem I've been struggling with or when I share a laugh with my classmates. \
Even though the sad moments weigh on me, the happiness I feel is a bit stronger. It's like finding a small flower blooming in the midst of a barren field, a tiny glimmer of light in the darkness. \
Despite the challenges, those moments of joy keep me going, reminding me that there's always something to smile about, even in the toughest of times."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet =  "During family gatherings, there are occasions when tensions run high, perhaps due to disagreements or unresolved conflicts. \
But amid the occasional conflicts, there are also moments of happiness, like when we share stories and laughter around the dinner table or reminisce about fond memories. \
Even though arguments may arise, the warmth of family bonds prevails. It's like finding harmony in the midst of discord, a glimmer of love amidst the strife. \
Despite the occasional disagreements, those moments of connection keep us united, reminding us of the strength of family ties."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet =  "At work, there are days when I feel overwhelmed, especially when deadlines are looming or when I face criticism from my boss. \
But in the midst of those stressful times, there are also moments of happiness, like when I successfully complete a project or receive recognition for my hard work. \
Even though the pressure gets to me, the joy of accomplishment outweighs the stress. It's like finding a silver lining in a stormy sky, a brief moment of relief amidst the chaos. \
Despite the challenges, those small victories keep me motivated, reminding me that perseverance pays off."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

## Neutral

In [ ]:
custom_tweet = \
"In the latest quarterly report, the company announced steady growth in revenue and market share,\
signaling stability in the face of economic uncertainty.However, analysts caution against complacency,\
highlighting potential challenges in the competitive landscape and evolving consumer preferences. \
Despite the cautious outlook, the company remains optimistic about its prospects for future growth."

preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()
preds_df.transpose()

In [ ]:
custom_tweet =  "Recent studies suggest that the new policy implemented by the government has elicited a mixed response from experts in the field. While some argue for its potential benefits in addressing socioeconomic disparities, others remain skeptical, citing potential unintended consequences. Despite the divergent opinions, further research is needed to assess the long-term implications of this policy shift."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet =  "In the latest quarterly report, the company announced steady growth in revenue and market share, signaling stability in the face of economic uncertainty. However, analysts caution against complacency, highlighting potential challenges in the competitive landscape and evolving consumer preferences. Despite the cautious outlook, the company remains optimistic about its prospects for future growth."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()

In [ ]:
custom_tweet = "The findings from the survey reveal a nuanced perspective on the issue, with respondents expressing a range of viewpoints. While some respondents voiced support for the proposed initiative, others expressed reservations, citing concerns about feasibility and implementation challenges. Despite the differing opinions, there is consensus on the need for further dialogue and collaboration to address the complex issues at hand."

print(custom_tweet)
preds = classifier(custom_tweet, return_all_scores=True)
preds_df = pd.DataFrame(preds[0])
plt.bar(labels_names_list, 100 * preds_df["score"], color='C0')
plt.ylabel("Class probability (%)")
plt.show()